# Investment Data Reconciliation: Walkthrough

This notebook runs the reconciliation one step at a time, with the reasoning next to each step.
Two systems should hold the same bond positions: the **custodian** (the bank that holds the securities) and the **investment accounting system** (where they're booked).
Every difference between them is a **break**. For each break, the job is to find it, explain its **root cause**, and log it in an **exception log**.

The production version of this logic lives in `src/reconcile.py`. The last cell checks that this notebook and the script produce identical exceptions.

In [1]:
import sys, io
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTPUT = ROOT / "data", ROOT / "output"
sys.path.insert(0, str(ROOT / "src"))

import generate_data
generate_data.main()   # fixed seed -> the same 200 positions and planted breaks every run

security_master.csv : 203 securities
custodian.csv       : 200 rows
accounting.csv      : 202 rows
breaks_truth.csv    : 46 planted items
break_type
decoy                        10
market_value_mismatch         8
accrued_interest_mismatch     6
id_format                     5
missing_in_accounting         4
rating_mismatch               4
duplicate_position            3
missing_at_custodian          3
timing_difference             3


## 1. Load and validate

**Why `dtype=str` on CUSIP matters.** A CUSIP is a 9-character ID and can start with zeros. If pandas (or Excel) decides a column is numeric, `007543127` silently becomes `7543127`, and every join on that ID fails without an error. Here's the trap in two lines:

In [2]:
fake_csv = "cusip\n007543127\n012345678\n"
print("default read :", pd.read_csv(io.StringIO(fake_csv)).cusip.tolist())
print("dtype=str    :", pd.read_csv(io.StringIO(fake_csv), dtype={"cusip": str}).cusip.tolist())

default read : [7543127, 12345678]
dtype=str    : ['007543127', '012345678']


In [3]:
def load(name):
    return pd.read_csv(DATA / name, dtype={"cusip": str, "position_id": str})

master = load("security_master.csv")
cust = load("custodian.csv")
acct = load("accounting.csv")

print(f"security_master: {len(master)} rows | custodian: {len(cust)} rows | accounting: {len(acct)} rows")
cust.head()

security_master: 203 rows | custodian: 200 rows | accounting: 202 rows


,position_id,cusip,par_value,market_value,book_value,accrued_interest,rating,as_of_date
0,P0096,1B0YQB108,19405000.0,21002236.44,18617819.69,637535.10,BBB-,2026-09-30
1,P0016,5EX5KE485,10935000.0,11825345.93,10608424.39,182060.16,AAA,2026-09-30
2,P0031,022659429,5795000.0,5275912.29,5863763.82,103525.26,BBB,2026-09-30
3,P0159,007660590,6060000.0,5887358.55,6021298.67,99990.00,BBB,2026-09-30
4,P0129,1KRJDG378,15700000.0,13497098.98,15941356.84,48081.25,A-,2026-09-30


In [4]:
validation = pd.DataFrame({
    "rows": [len(cust), len(acct)],
    "nulls": [cust.isna().sum().sum(), acct.isna().sum().sum()],
    "cusips_not_9_chars": [(cust.cusip.str.len() != 9).sum(), (acct.cusip.str.len() != 9).sum()],
    "as_of_dates": [cust.as_of_date.value_counts().to_dict(), acct.as_of_date.value_counts().to_dict()],
}, index=["custodian", "accounting"])
validation

,rows,nulls,cusips_not_9_chars,as_of_dates
custodian,200,0,0,{'2026-09-30': 200}
accounting,202,0,5,"{'2026-09-30': 199, '2026-09-29': 3}"


Validation already surfaces two problems before any matching:

- accounting has **5 CUSIPs that aren't 9 characters**
- accounting has **3 rows with a different as-of date**

It also has 2 more rows than the custodian, even though some positions are missing, which points to duplicates.

## 2. Normalize IDs

An all-digit CUSIP shorter than 9 characters is a number that lost its leading zeros somewhere upstream. `zfill(9)` pads it back. Each fix is **logged as a break**, not silently repaired, because the upstream process that stripped the zeros is still broken and needs an owner.

In [5]:
exceptions = []

def log(cusip, break_type, field, cust_val, acct_val, diff=None):
    exceptions.append(dict(cusip=cusip, break_type=break_type, field=field,
                           custodian_value=cust_val, accounting_value=acct_val, difference=diff))

broken = acct.cusip.str.isdigit() & (acct.cusip.str.len() < 9)
fixes = acct.loc[broken, ["position_id", "cusip"]].assign(fixed=lambda d: d.cusip.str.zfill(9))
for r in fixes.itertuples():
    log(r.fixed, "id_format", "cusip", r.fixed, r.cusip)

acct.loc[broken, "cusip"] = acct.loc[broken, "cusip"].str.zfill(9)
fixes

,position_id,cusip,fixed
27,P0041,79547311,079547311
36,P0060,99062127,099062127
77,P0191,78639663,078639663
140,P0025,5516810,005516810
153,P0078,9712001,009712001


## 3. Duplicates

Checked **before** matching. If a row loaded twice, the merge would produce a double-counted position.

In [6]:
dup_counts = acct.cusip.value_counts()
dups = dup_counts[dup_counts > 1]
for cusip, n in dups.items():
    log(cusip, "duplicate_position", "row_count", None, n)

acct = acct.drop_duplicates(subset="cusip", keep="first")
dups.rename("rows_in_accounting").to_frame()

,rows_in_accounting
cusip,
633DFH210,2
6XJ3GA403,2
2HJ8HC679,2


## 4. Match

An **outer** merge keeps every position from both sides. `indicator=True` adds a `_merge` column saying where each row came from:

- `both`: matched
- `left_only`: at the custodian but **missing in accounting** (trade not booked, feed failure)
- `right_only`: in accounting but **missing at the custodian** (pending settlement, stale position)

In [7]:
merged = cust.merge(acct, on="cusip", how="outer", suffixes=("_cust", "_acct"), indicator=True)
print(merged["_merge"].value_counts().to_dict())

for c in merged.loc[merged._merge == "left_only", "cusip"]:
    log(c, "missing_in_accounting", "position", "present", "absent")
for c in merged.loc[merged._merge == "right_only", "cusip"]:
    log(c, "missing_at_custodian", "position", "absent", "present")

both = merged[merged._merge == "both"].copy()
merged.loc[merged._merge != "both", ["cusip", "position_id_cust", "position_id_acct", "_merge"]]

{'both': 196, 'left_only': 4, 'right_only': 3}


,cusip,position_id_cust,position_id_acct,_merge
2,001702562,P0036,NaN,left_only
65,070958004,NaN,P0203,right_only
93,1VQ65E039,P0082,NaN,left_only
120,3GJSPG355,P0062,NaN,left_only
123,3MVKFB290,P0050,NaN,left_only
141,55HKMD463,NaN,P0201,right_only
194,9F173G954,NaN,P0202,right_only


## 5. Timing (checked before values)

If accounting loaded **yesterday's** file, its accrued interest will be off by a day. That's not a separate accrued-interest break: the root cause is **timing**. So date mismatches are flagged first, and those positions are left out of the value comparison. Otherwise one problem gets logged twice under the wrong cause.

In [8]:
timing = both.as_of_date_cust != both.as_of_date_acct
for r in both[timing].itertuples():
    log(r.cusip, "timing_difference", "as_of_date", r.as_of_date_cust, r.as_of_date_acct)

both.loc[timing, ["cusip", "as_of_date_cust", "as_of_date_acct", "accrued_interest_cust", "accrued_interest_acct"]]

,cusip,as_of_date_cust,as_of_date_acct,accrued_interest_cust,accrued_interest_acct
86,1GLYHC516,2026-09-30,2026-09-29,160037.50,157456.25
119,3AXE8H491,2026-09-30,2026-09-29,1002099.83,996470.06
145,5FMTBG248,2026-09-30,2026-09-29,502879.03,497412.95


## 6. Compare values within tolerance

A difference is a break only if it beats **`max($1.00, 0.01% of the value)`**:

- **Absolute floor ($1):** stops rounding noise on small numbers. A 40-cent gap on $500 of accrued interest is 0.08%, above the relative tolerance, but it isn't a real break.
- **Relative part (0.01%):** stops rounding noise on big numbers. On a $20M position, 0.01% is $2,000.

With only one of the two, one of those cases raises false alarms. The table below shows every non-zero difference and how the tolerance handled it. The small ones are the planted rounding **decoys**.

In [9]:
ABS_TOL, REL_TOL = 1.00, 0.0001
comparable = both[~timing]
rows = []
for field in ["market_value", "book_value", "accrued_interest"]:
    c, a = comparable[f"{field}_cust"], comparable[f"{field}_acct"]
    diff = a - c
    tol = (c.abs() * REL_TOL).clip(lower=ABS_TOL)
    for idx in comparable.index[diff.abs() > 0]:
        is_break = abs(diff[idx]) > tol[idx]
        rows.append(dict(cusip=comparable.at[idx, "cusip"], field=field, custodian=c[idx], accounting=a[idx],
                         difference=round(diff[idx], 2), tolerance=round(tol[idx], 2), is_break=is_break))
        if is_break:
            log(comparable.at[idx, "cusip"], f"{field}_mismatch", field, round(c[idx], 2), round(a[idx], 2), round(diff[idx], 2))

diffs = pd.DataFrame(rows).sort_values(["is_break", "field"], ascending=[False, True])
print(diffs.is_break.value_counts().rename({True: "breaks", False: "within tolerance (ignored)"}).to_dict())
diffs

{'breaks': 14, 'within tolerance (ignored)': 10}


,cusip,field,custodian,accounting,difference,tolerance,is_break
17,1C80XH200,accrued_interest,66483.33,80334.02,13850.69,6.65,True
18,35N5EG273,accrued_interest,477442.71,488418.40,10975.69,47.74,True
19,3J6FVA746,accrued_interest,92584.38,96890.63,4306.25,9.26,True
20,3R3GMD628,accrued_interest,314058.58,330983.89,16925.31,31.41,True
21,4LA75B179,accrued_interest,93638.65,104988.79,11350.14,9.36,True
22,5524NG137,accrued_interest,187381.25,191396.56,4015.31,18.74,True
0,057948986,market_value,4351223.56,4286426.96,-64796.60,435.12,True
2,097889182,market_value,16826520.63,15988186.15,-838334.48,1682.65,True
3,25EL7C359,market_value,15375133.10,15544885.27,169752.17,1537.51,True
4,3AHUDD017,market_value,14113070.18,13621982.77,-491087.41,1411.31,True


## 7. Ratings vs. the security master

The **security master** is the golden reference for security attributes. If one system's rating disagrees with it, that system's reference data is stale. This matters to an insurer because ratings drive capital charges.

In [10]:
ref = master[["cusip", "rating"]].rename(columns={"rating": "rating_master"})
rated = both.merge(ref, on="cusip", how="left")
bad = rated[(rated.rating_acct != rated.rating_master) | (rated.rating_cust != rated.rating_master)]
for r in bad.itertuples():
    log(r.cusip, "rating_mismatch", "rating", r.rating_cust, r.rating_acct)

bad[["cusip", "rating_cust", "rating_acct", "rating_master"]]

,cusip,rating_cust,rating_acct,rating_master
3,002871325,A+,A,A+
55,041764317,A-,BBB+,A-
109,2ZAAAD140,A-,BBB+,A-
121,3RZ0ZA539,A-,BBB+,A-


## 8. Exception log

In [11]:
exc = pd.DataFrame(exceptions)
exc.insert(0, "exception_id", [f"EXC-{i:04d}" for i in range(1, len(exc) + 1)])
exc["status"] = "Open"
exc

,exception_id,cusip,break_type,field,custodian_value,accounting_value,difference,status
0,EXC-0001,079547311,id_format,cusip,079547311,79547311,NaN,Open
1,EXC-0002,099062127,id_format,cusip,099062127,99062127,NaN,Open
2,EXC-0003,078639663,id_format,cusip,078639663,78639663,NaN,Open
3,EXC-0004,005516810,id_format,cusip,005516810,5516810,NaN,Open
4,EXC-0005,009712001,id_format,cusip,009712001,9712001,NaN,Open
5,EXC-0006,633DFH210,duplicate_position,row_count,None,2,NaN,Open
6,EXC-0007,6XJ3GA403,duplicate_position,row_count,None,2,NaN,Open
7,EXC-0008,2HJ8HC679,duplicate_position,row_count,None,2,NaN,Open
8,EXC-0009,001702562,missing_in_accounting,position,present,absent,NaN,Open
9,EXC-0010,1VQ65E039,missing_in_accounting,position,present,absent,NaN,Open


## 9. Summary

In [12]:
all_positions = set(merged.cusip)
clean = len(all_positions - set(exc.cusip))
mv = exc[exc.break_type == "market_value_mismatch"]

print(f"Unique positions:            {len(all_positions)}")
print(f"Reconciled cleanly:          {clean}")
print(f"Match rate:                  {clean / len(all_positions):.1%}")
print(f"MV break impact (gross/net): ${mv.difference.abs().sum():,.2f} / ${mv.difference.sum():,.2f}")
exc.break_type.value_counts().rename("count").to_frame()

Unique positions:            203
Reconciled cleanly:          167
Match rate:                  82.3%
MV break impact (gross/net): $3,174,061.56 / $-1,518,185.56


,count
break_type,
market_value_mismatch,8
accrued_interest_mismatch,6
id_format,5
missing_in_accounting,4
rating_mismatch,4
duplicate_position,3
missing_at_custodian,3
timing_difference,3


## 10. Grade against the answer key

`breaks_truth.csv` lists every break the generator planted, plus 10 decoys that should **not** be flagged.

In [13]:
truth = pd.read_csv(DATA / "breaks_truth.csv", dtype={"cusip": str})
planted = truth[truth.break_type != "decoy"]
expected = set(zip(planted.cusip, planted.break_type))
found = set(zip(exc.cusip, exc.break_type))
decoys = set(truth.loc[truth.break_type == "decoy", "cusip"])

print(f"Recall:          {len(expected & found)}/{len(expected)}")
print(f"False positives: {len(found - expected)}")
print(f"Decoys flagged:  {len(decoys & set(exc.cusip))} of {len(decoys)}")

Recall:          36/36
False positives: 0
Decoys flagged:  0 of 10


## 11. Cross-check: notebook vs. `src/reconcile.py` vs. SQL

In [14]:
import reconcile, run_sql
reconcile.exceptions.clear()
reconcile.main()
script_log = pd.read_csv(OUTPUT / "exception_log.csv", dtype={"cusip": str})

same = set(zip(script_log.cusip, script_log.break_type)) == found
print("\nNotebook and script produce identical exceptions:", same)

=== 1. Load + validate ===

[security_master] rows=203
  nulls: none
  CUSIPs not 9 characters: 0

[custodian] rows=200
  nulls: none
  negative values: 0
  CUSIPs not 9 characters: 0
  as_of_date values: {'2026-09-30': 200}

[accounting] rows=202
  nulls: none
  negative values: 0
  CUSIPs not 9 characters: 5
  as_of_date values: {'2026-09-30': 199, '2026-09-29': 3}

=== 2. Normalize IDs ===


  re-padded CUSIPs: 5

=== 3. Duplicates ===
  duplicate CUSIPs: 3

=== 4. Match ===
  {'both': 196, 'left_only': 4, 'right_only': 3}

=== 7. As-of date (checked before values: timing explains value diffs) ===
  timing differences: 3

=== 5. Compare values (abs + relative tolerance) ===
  market_value      breaks= 8  non-zero diffs within tolerance=2
  book_value        breaks= 0  non-zero diffs within tolerance=5
  accrued_interest  breaks= 6  non-zero diffs within tolerance=3

=== 6. Security master (ratings) ===
  rating mismatches: 4  positions missing from master: 0

=== 8/9. Output ===
    section                         name       value
break_count        market_value_mismatch           8
break_count    accrued_interest_mismatch           6
break_count                    id_format           5
break_count        missing_in_accounting           4
break_count              rating_mismatch           4
break_count           duplicate_position           3
break_count         missing_at

In [15]:
run_sql.main()

-- id_format: 5 rows
-- duplicate_position: 3 rows
-- missing_in_accounting: 4 rows
-- missing_at_custodian: 3 rows
-- timing_difference: 3 rows
-- value_mismatch: 14 rows
-- rating_mismatch: 4 rows

=== Python vs SQL ===
               break_type  python  sql  match
accrued_interest_mismatch       6    6   True
       duplicate_position       3    3   True
                id_format       5    5   True
    market_value_mismatch       8    8   True
     missing_at_custodian       3    3   True
    missing_in_accounting       4    4   True
          rating_mismatch       4    4   True
        timing_difference       3    3   True

MV dollar impact (SQL): abs=3,174,061.56  net=-1,518,185.56
ALL MATCH
